In [16]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [17]:
def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def solvenominal (sets,p,R,r,m,r_f,c,ordering):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    a = cp.Variable(I)
    constraints = [a>= 0, a<=1, cp.sum(a)<= 1]
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value,prob.value)

In [22]:
def cut_plane(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [a>= 0, a<=1, cp.sum(a)<= 1]
    h = np.zeros(N)
    iterations = 1
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    [rbvalue,q_b,obj_value] = robustcheck(w,R,r,p,m,r_f)
    nonstop = True
    while nonstop:
        print(rbvalue)
        if rbvalue <= c+1e-5:
            return(w,obj_value,iterations)
        h = q_b
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
        obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        [rbvalue,q_b,obj_value] = robustcheck(w,R,r,p,m,r_f)
        iterations = iterations + 1
        print(iterations)
    

In [7]:
np.random.seed(5)

In [20]:
N=10
p = np.zeros(N)+1/N
I = 2
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))

[0.05103441 0.07753556]


In [23]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.28
print(cut_plane(R,r,c,p,m,r_f))

0.12137063078333064
(array([4.07665487e-11, 1.00000000e+00]), 0.12137063078335304, 1)
